**Widgets**

In [1]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("Widgets").getOrCreate()

In [4]:
from google.colab import files, output
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

uploaded=files.upload()
file_name=next(iter(uploaded))
df=pd.read_csv(file_name)
print("Dataset loaded successfully!")
print("Number of records:",len(df))
display(df.head())



Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Dataset loaded successfully!
Number of records: 12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [7]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]

# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)
# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [8]:
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))


In [9]:
column_dropdown = widgets.Dropdown(
    options=list(df.columns),
    description="Column:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

display(column_dropdown)

Dropdown(description='Column:', layout=Layout(width='300px'), options=('production_id', 'production_date', 'pl…

**Working with DataFrame**

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [11]:
# Transformation 1
filtered_df = df.filter(col("quantity")>3)
print("Transformation 1: Filtered DataFrame")
filtered_df.show()
# Transformation 2
selected_df=filtered_df.select("city", "product", "price")
print("Transformation 2: Selected DataFrame")

#action
selected_df.show()

Transformation 1: Filtered DataFrame
+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      2|Bangalore| Mobile|       5|20000|
|      4|Hyderabad| Mobile|       4|22000|
|      6|Bangalore| Tablet|       6|12000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

Transformation 2: Selected DataFrame
+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|Bangalore| Mobile|20000|
|Hyderabad| Mobile|22000|
|Bangalore| Tablet|12000|
|   Mumbai| Mobile|18000|
|Hyderabad| Tablet|14000|
|   Mumbai| Tablet|13000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [13]:
# transformation 1: Filter sales
step1 = df.filter(col("quantity") > 3)
print("Transformation 1: Filtered DataFrame")
# Transformation 2: Calculate revenue
step2 = step1.withColumn("revenue", col("quantity") * col("price"))
print("Transformation 2: withcolumn DataFrame")
# Transformation 3: Group by city
step3 = step2.groupBy("city").agg(sum("revenue").alias("total_revenue"))
print("Transformation 3: Grouped DataFrame")
# Transformation 4 : Sort results
step4 = step3.orderBy(desc("total_revenue"))
print("Transformation 4: Sorted DataFrame")
# Action: Show the result
step4.show()

Transformation 1: Filtered DataFrame
Transformation 2: withcolumn DataFrame
Transformation 3: Grouped DataFrame
Transformation 4: Sorted DataFrame
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|   Mumbai|       235000|
|Bangalore|       172000|
|Hyderabad|       158000|
|  Chennai|       126000|
+---------+-------------+



In [14]:
# Define Transformation
result=df.filter(col("price")>20000).select("city","product","price").orderBy(desc("price"))
# Sparks execution plan
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [price#4L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(price#4L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=186]
      +- Project [city#1, product#2, price#4L]
         +- Filter (isnotnull(price#4L) AND (price#4L > 20000))
            +- Scan ExistingRDD[sale_id#0L,city#1,product#2,quantity#3L,price#4L]




In [15]:
result.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [16]:
high_value_sales =df.filter(col("price") > 20000)
#Action 1
high_value_sales.show()

#action 2
print("Total Records:",high_value_sales.count())

# Action 3
records=high_value_sales.collect()
print(records)

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      7|  Chennai| Laptop|       2|58000|
|     10|Bangalore| Laptop|       3|52000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

Total Records: 6
[Row(sale_id=1, city='Hyderabad', product='Laptop', quantity=2, price=55000), Row(sale_id=4, city='Hyderabad', product='Mobile', quantity=4, price=22000), Row(sale_id=5, city='Mumbai', product='Laptop', quantity=1, price=60000), Row(sale_id=7, city='Chennai', product='Laptop', quantity=2, price=58000), Row(sale_id=10, city='Bangalore', product='Laptop', quantity=3, price=52000), Row(sale_id=12, city='Chennai', product='Mobile', quantity=6, price=21000)]


**Practice Assignment**

In [17]:
# Upload the three CSV files to Google Colab
from google.colab import files

uploaded = files.upload()

Saving lazy_execution_products.csv to lazy_execution_products.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_stores.csv to lazy_execution_stores.csv


In [18]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LazyExecutionExercises").getOrCreate()

sales_df = spark.read.csv(
    "lazy_execution_sales_25000.csv", header=True, inferSchema=True
)
products_df = spark.read.csv(
    "lazy_execution_products.csv", header=True, inferSchema=True
)
stores_df = spark.read.csv(
    "lazy_execution_stores.csv", header=True, inferSchema=True
)

In [19]:
print("Sales:", sales_df.columns)
print("Products:", products_df.columns)
print("Stores:", stores_df.columns)

Sales: ['order_id', 'order_date', 'store_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'order_status', 'sales_channel']
Products: ['product_id', 'product_name', 'category', 'list_price', 'cost_price']
Stores: ['store_id', 'city', 'region', 'store_type']


**Lazy Execution in Pyspark**

**Part A — Basic transformations and actions**

In [20]:
# 1. Filter all transactions where quantity > 5. Store the result in a new DataFrame without displaying it.
filtered_df = sales_df.filter(sales_df.quantity > 5)

In [21]:
# 2. Select only order_id, product_id, quantity, and unit_price. Use show() to trigger execution.
sales_df.select("order_id", "product_id", "quantity", "unit_price").show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000011|      P007|       1|     79999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000016|      P006|       1|     55999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
+--------+----------+--------+----

In [22]:
# 3. Filter transactions with unit_price > 20000. Use count() to determine the number of matching records.
sales_df.filter(sales_df.unit_price > 20000).count()

6560

In [25]:
# 4. Create a new column gross_amount by multiplying quantity * unit_price. Inspect the execution plan using explain() before displaying the results.
gross_df = sales_df.withColumn(
    "gross_amount", sales_df.quantity * sales_df.unit_price
)
gross_df.explain()

== Physical Plan ==
*(1) Project [order_id#137, order_date#138, store_id#139, product_id#140, quantity#141, unit_price#142, discount_pct#143, payment_method#144, order_status#145, sales_channel#146, (quantity#141 * unit_price#142) AS gross_amount#313]
+- FileScan csv [order_id#137,order_date#138,store_id#139,product_id#140,quantity#141,unit_price#142,discount_pct#143,payment_method#144,order_status#145,sales_channel#146] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...




In [26]:
# 5. Filter transactions with order_status = "Completed" and sort them by unit_price descending. Trigger execution using take(10).
sales_df.filter(sales_df.order_status == "Completed").orderBy(
    sales_df.unit_price.desc()
).take(10)

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9

**Part B — Chained transformations**

In [28]:
# 6. Create a pipeline that filters completed orders, calculates gross amount, selects five relevant columns, and sorts by gross amount descending. Use only one action at the end.
(
    sales_df.filter(sales_df.order_status == "Completed")
    .withColumn("gross_amount", sales_df.quantity * sales_df.unit_price)
    .select(
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "gross_amount",
    )
    .orderBy(col("gross_amount").desc())

)

DataFrame[order_id: string, product_id: string, quantity: int, unit_price: int, gross_amount: int]

In [29]:
# 7. Filter orders with discounts of at least 10%, calculate the discounted sales amount, and display the top 15 orders.
from pyspark.sql.functions import col, when

# Normalizes discount whether stored as 10 or 0.10
discount_factor = when(col("discount_pct") > 1, col("discount_pct") / 100).otherwise(col("discount_pct"))

(
    sales_df.filter((col("discount_pct") >= 10) | (col("discount_pct") >= 0.10))
    .withColumn("discounted_amount", (col("quantity") * col("unit_price")) * (1 - discount_factor))

)

DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string, discounted_amount: double]

In [30]:
# 8. Group transactions by payment_method , calculate total quantity sold, and sort the results in descending order.
from pyspark.sql.functions import desc, sum

sales_df.groupBy("payment_method").agg(
    sum("quantity").alias("total_quantity")
).orderBy(desc("total_quantity"))

DataFrame[payment_method: string, total_quantity: bigint]

In [31]:
# 9. Group transactions by sales_channel and order_status , then calculate the transaction count for each combination.
from pyspark.sql.functions import count

sales_df.groupBy("sales_channel", "order_status").agg(
    count("*").alias("transaction_count")
)

DataFrame[sales_channel: string, order_status: string, transaction_count: bigint]

In [32]:
# 10. Create a DataFrame containing only order_id , store_id , product_id , and quantity . Use drop() to remove store_id , then use first() to retrieve one record.
result = (
    sales_df.select("order_id", "store_id", "product_id", "quantity")
    .drop("store_id")
    .first()
)
print(result)

Row(order_id='O000001', product_id='P002', quantity=3)


**Part C — Joins and aggregations**

In [34]:
# 11. Join sales_df with products_df using product_id . Select product name, category, quantity, and unit price. Display 20 records.
sales_df.join(products_df, "product_id").select(
    "product_name", "category", "quantity", "unit_price"
)

DataFrame[product_name: string, category: string, quantity: int, unit_price: int]

In [35]:
# 12. Join sales_df with stores_df using store_id . Group by city and calculate the total quantity sold.
from pyspark.sql.functions import sum

sales_df.join(stores_df, "store_id").groupBy("city").agg(
    sum("quantity").alias("total_quantity")
)

DataFrame[city: string, total_quantity: bigint]

In [36]:
# 13. Join all three DataFrames. Calculate total completed-order revenue for each product category and region.
from pyspark.sql.functions import col, sum

(
    sales_df.filter(col("order_status") == "Completed")
    .join(products_df, "product_id")
    .join(stores_df, "store_id")
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .groupBy("category", "region")
    .agg(sum("revenue").alias("total_revenue"))
)

DataFrame[category: string, region: string, total_revenue: bigint]

In [37]:
# 14. Filter completed transactions, join the product master, calculate discounted revenue, and display the top five products by total revenue.
from pyspark.sql.functions import col, desc, sum, when

discount_factor = when(col("discount_pct") > 1, col("discount_pct") / 100).otherwise(col("discount_pct"))

(
    sales_df.filter(col("order_status") == "Completed")
    .join(products_df, "product_id")
    .withColumn("discounted_revenue", (col("quantity") * col("unit_price")) * (1 - discount_factor))
    .groupBy("product_name")
    .agg(sum("discounted_revenue").alias("total_revenue"))
    .orderBy(desc("total_revenue"))
)

DataFrame[product_name: string, total_revenue: double]

**Part D — Understanding lazy execution**

In [38]:
# 15. Create five consecutive transformations using filter() , select() , withColumn() , groupBy() , and orderBy() . Add a Python print() after each transformation. Observe when Spark actually processes the data.
from pyspark.sql.functions import col, sum

t1 = sales_df.filter(col("quantity") > 2)
print("1. filter() registered")

t2 = t1.select("product_id", "quantity", "unit_price")
print("2. select() registered")

t3 = t2.withColumn("gross_amount", col("quantity") * col("unit_price"))
print("3. withColumn() registered")

t4 = t3.groupBy("product_id").agg(sum("gross_amount").alias("total_sales"))
print("4. groupBy() registered")

t5 = t4.orderBy(col("total_sales").desc())
print("5. orderBy() registered")

print("\n Triggering Action:")
t5.show(5)

1. filter() registered
2. select() registered
3. withColumn() registered
4. groupBy() registered
5. orderBy() registered

--- Triggering Action: show() ---
+----------+-----------+
|product_id|total_sales|
+----------+-----------+
|      P007|  599112511|
|      P006|  427944358|
|      P014|  231502283|
|      P008|  170146602|
|      P004|  125197635|
+----------+-----------+
only showing top 5 rows


In [40]:
# 16. Create a filtered DataFrame and inspect it with explain() . Then call show() and count() separately. Observe the Spark jobs in the Colab output.
from pyspark.sql.functions import col

df_filtered = sales_df.filter(col("quantity") > 5)

print("Execution Plan")
df_filtered.explain()

print("\nTriggering Action: show()")
df_filtered.show(5)

print("\nTriggering Action: count()")
print("Count:", df_filtered.count())

Execution Plan
== Physical Plan ==
*(1) Filter (isnotnull(quantity#141) AND (quantity#141 > 5))
+- FileScan csv [order_id#137,order_date#138,store_id#139,product_id#140,quantity#141,unit_price#142,discount_pct#143,payment_method#144,order_status#145,sales_channel#146] Batched: false, DataFilters: [isnotnull(quantity#141), (quantity#141 > 5)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,5)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...



Triggering Action: show()
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+--------

In [41]:
# 17. Use collect() to retrieve the grouped total quantity for each product category. Explain why collecting a small aggregated result is safer than collecting all 25,000 rows.
from pyspark.sql.functions import sum

result = (
    sales_df.join(products_df, "product_id")
    .groupBy("category")
    .agg(sum("quantity").alias("total_quantity"))
    .collect()
)
print(result)


[Row(category='Computers', total_quantity=16258), Row(category='Displays', total_quantity=7870), Row(category='Mobiles', total_quantity=16572), Row(category='Accessories', total_quantity=41839), Row(category='Storage', total_quantity=17251), Row(category='Audio', total_quantity=17159), Row(category='Tablets', total_quantity=7915)]


In [42]:
# 18. Filter completed orders, calculate discounted revenue, and save the resulting DataFrame as Parquet using write.mode("overwrite").parquet() . Identify which operation triggers execution.
from pyspark.sql.functions import col, when

discount_factor = when(col("discount_pct") > 1, col("discount_pct") / 100).otherwise(col("discount_pct"))

final_df = sales_df.filter(col("order_status") == "Completed").withColumn(
    "discounted_revenue",
    (col("quantity") * col("unit_price")) * (1 - discount_factor),
)
final_df.write.mode("overwrite").parquet("output_sales.parquet")
print("Saved to Parquet successfully.")

Saved to Parquet successfully.


**Caching**

In [43]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("SalesAnalysis").getOrCreate()
uploaded=files.upload()
dataframes={}
for filename in uploaded.keys():
  df=spark.read.csv(filename,header=True,inferSchema=True)
  dataframes[filename]=df
  print(f"\nFile: {filename}")
  print("Records:",df.count())
  df.show(5)

Saving lazy_execution_stores (1).csv to lazy_execution_stores (1).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1).csv
Saving lazy_execution_products (1).csv to lazy_execution_products (1).csv

File: lazy_execution_stores (1).csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows

File: lazy_execution_sales_25000 (1).csv
Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+---

In [44]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

Sales file selected: lazy_execution_sales_25000 (1).csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Comple

In [45]:
# Transformation
completed_df = sales_df.filter(
    F.col("order_status") == "Completed"
)
print("Transformation created")
# Action 1
print("Completed Orders:",completed_df.count())

# Action 2
completed_df.show(10)

Transformation created
Completed Orders: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04

In [46]:
# Cache the filtered DataFrame
completed_df.cache()
print("Is Cached:",completed_df.is_cached)

# First action populates cache
completed_df.count()

# Reuse cache data
completed_df.show(10)

# Another Action using cached data
completed_df.groupBy("product_id").count().show()


Is Cached: True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       

In [47]:
#create filtered DataFrame
high_quantity_df =sales_df.filter(
    F.col("quantity")>5
)
# Persist using Memory_and_disk
high_quantity_df.persist(StorageLevel.MEMORY_AND_DISK)

# Trigger execution
high_quantity_df.count()

# Reuse persisted data
high_quantity_df.show(10)

# check storage level
print(high_quantity_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [48]:
# check cache status
print("Before:",completed_df.is_cached)
# Remove cache
completed_df.unpersist(blocking=True)
#check again
print("After:",completed_df.is_cached)

#Remove the other presisted dataframe
high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [49]:
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [51]:
# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 0.858 seconds
Second Execution: 0.065 seconds


**Partition**

In [52]:
# Check total records
print("Total Records:",sales_df.count())

#check number of partitions
print("Number of Partitions:",sales_df.rdd.getNumPartitions())


Total Records: 25000
Number of Partitions: 1


In [54]:
#create 8 Partitions
partitioned_df = sales_df.repartition(8)
print("Original Partitions:",sales_df.rdd.getNumPartitions())
print("New Partitions:",partitioned_df.rdd.getNumPartitions())
#Trigger execution
partitioned_df.count()


Original Partitions: 1
New Partitions: 8


25000

In [55]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



In [56]:
# Key Based Partition
# Partition by store_id
store_partitioned_df = sales_df.repartition(8,"store_id")
print("Original Partitions:",sales_df.rdd.getNumPartitions())
print("New Partitions:",store_partitioned_df.rdd.getNumPartitions())

# Inspect partition distribution
store_partitioned_df.withColumn("partition_id",F.spark_partition_id()).groupBy("partition_id","store_id").count().orderBy("partition_id","store_id").show(30)

Original Partitions: 1
New Partitions: 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



In [57]:
# Reducing the Partitions
# Repartition
#create 8 partition
df8=sales_df.repartition(8)

# Reduce using repartition
df_repartition=df8.repartition(3)

# Reduce using coalesce
df_coalesce=df8.coalesce(3)

print("Repartition:",df_repartition.rdd.getNumPartitions())
print("Coalesce:",df_coalesce.rdd.getNumPartitions())

# Examine execution plans
print("Repartition Execution Plan")
df_repartition.explain()

print("Coalesce Execution Plan")
df_coalesce.explain()

Repartition: 3
Coalesce: 3
Repartition Execution Plan
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=2179]
         +- FileScan csv [order_id#683,order_date#684,store_id#685,product_id#686,quantity#687,unit_price#688,discount_pct#689,payment_method#690,order_status#691,sales_channel#692] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000 (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=2179]
   +- FileScan csv [order_id#683,order_date#684,store_id#685,product_id#686,quantity#687,unit_price#688,discount_pct#689,payment_method#690,order_status#691,sales_channel#692] B

In [58]:
# Find the current number of partitions
print("Partition:",sales_df.rdd.getNumPartitions())

Partition: 1


In [59]:
# Increase the number of partitions to 12
df12=sales_df.repartition(12)
print("Partition:",df12.rdd.getNumPartitions())


Partition: 12


In [63]:
# Display the number of records in each partition
df12.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy(
    "partition_id"
).show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+



In [64]:
# Repartition the dataframe into 6 partition using product_id
df6=sales_df.repartition(6,"product_id")
print("Partition:",df6.rdd.getNumPartitions())



Partition: 6


In [66]:
#Reduce 12 partitions to using 4 coalesce
df4=df12.coalesce(4)
print("Partition:",df4.rdd.getNumPartitions())

Partition: 4
